# Lab 16: Canales Cuánticos y Operadores de Kraus

La evolución de un sistema cuántico abierto (acoplado a un entorno) no es unitaria: el estado mixto $\rho$ evoluciona mediante un **canal cuántico** $\mathcal{E}$:

$$\mathcal{E}(\rho) = \sum_k K_k \rho K_k^\dagger$$

donde $\{K_k\}$ son los **operadores de Kraus**, que satisfacen $\sum_k K_k^\dagger K_k = I$ (conservación de traza).

Estudiaremos los tres canales más importantes: despolarizante, amortiguamiento de amplitud y amortiguamiento de fase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit.quantum_info import DensityMatrix, Kraus, SuperOp, state_fidelity, entropy
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, amplitude_damping_error, phase_damping_error, depolarizing_error

I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Y = np.array([[0,-1j],[1j,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)
plus = np.array([1,1],dtype=complex)/np.sqrt(2)
rho_plus = np.outer(plus, plus.conj())
rho_0 = np.array([[1,0],[0,0]], dtype=complex)

print("Estados de prueba preparados: ρ₊ = |+⟩⟨+|  y  ρ₀ = |0⟩⟨0|")

## 1. Canal Despolarizante

El canal despolarizante con parámetro $p$ sustituye el estado por la mezcla máxima con probabilidad $p$:

$$\mathcal{E}_p(\rho) = (1-p)\rho + \frac{p}{3}(X\rho X + Y\rho Y + Z\rho Z) = \left(1-\frac{4p}{3}\right)\rho + \frac{p}{3}I/2$$

Sus operadores de Kraus son $K_0=\sqrt{1-p}\,I$, $K_1=\sqrt{p/3}\,X$, $K_2=\sqrt{p/3}\,Y$, $K_3=\sqrt{p/3}\,Z$.

In [ ]:
def depolarizing_kraus(p):
    return [np.sqrt(1-p)*I, np.sqrt(p/3)*X, np.sqrt(p/3)*Y, np.sqrt(p/3)*Z]

def apply_kraus(rho, kraus_ops):
    return sum(K @ rho @ K.conj().T for K in kraus_ops)

p_vals = np.linspace(0, 1, 100)
purities_dep = []
for p in p_vals:
    K = depolarizing_kraus(p)
    rho_out = apply_kraus(rho_plus, K)
    dm = DensityMatrix(rho_out)
    purities_dep.append(float(dm.purity()))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(p_vals, purities_dep, color='steelblue', lw=2)
axes[0].axhline(0.5, color='gray', ls=':', lw=1.5, label='Mezcla máxima')
axes[0].set_title('Canal Despolarizante')
axes[0].set_xlabel('p'); axes[0].set_ylabel('Pureza Tr(ρ²)')
axes[0].legend()

# Verificación operadores Kraus
K_dep = depolarizing_kraus(0.3)
completeness = sum(K.conj().T @ K for K in K_dep)
print(f"Suma Kᵢ†Kᵢ (debe ser I): error = {np.linalg.norm(completeness - I):.2e}")

# Acción sobre la esfera de Bloch
for p_test in [0.0, 0.25, 0.5, 0.75]:
    rho_out = apply_kraus(rho_plus, depolarizing_kraus(p_test))
    dm = DensityMatrix(rho_out)
    print(f"p={p_test:.2f}: pureza={float(dm.purity()):.4f}, S(ρ)={float(entropy(dm,base=2)):.4f} bits")

# Continuar los otros dos canales en la misma figura
def amplitude_damping_kraus(gamma):
    K0 = np.array([[1,0],[0,np.sqrt(1-gamma)]], dtype=complex)
    K1 = np.array([[0,np.sqrt(gamma)],[0,0]], dtype=complex)
    return [K0, K1]

def phase_damping_kraus(lam):
    K0 = np.array([[1,0],[0,np.sqrt(1-lam)]], dtype=complex)
    K1 = np.array([[0,0],[0,np.sqrt(lam)]], dtype=complex)
    return [K0, K1]

purities_amp  = [float(DensityMatrix(apply_kraus(rho_plus, amplitude_damping_kraus(g))).purity()) for g in p_vals]
purities_phase = [float(DensityMatrix(apply_kraus(rho_plus, phase_damping_kraus(lam))).purity()) for lam in p_vals]

axes[1].plot(p_vals, purities_amp, color='darkorange', lw=2)
axes[1].axhline(0.5, color='gray', ls=':', lw=1.5)
axes[1].set_title('Amortiguamiento de Amplitud')
axes[1].set_xlabel('γ'); axes[1].set_ylabel('Pureza Tr(ρ²)')

axes[2].plot(p_vals, purities_phase, color='mediumseagreen', lw=2)
axes[2].axhline(0.5, color='gray', ls=':', lw=1.5)
axes[2].set_title('Amortiguamiento de Fase (Decoherencia)')
axes[2].set_xlabel('λ'); axes[2].set_ylabel('Pureza Tr(ρ²)')

plt.tight_layout(); plt.show()

## 2. Amortiguamiento de Amplitud (Relajación T₁)

El amortiguamiento de amplitud modela la **relajación energética**: el qubit decae del estado excitado $|1\rangle$ al estado base $|0\rangle$ con probabilidad $\gamma = 1 - e^{-t/T_1}$.

$$K_0 = \begin{pmatrix}1 & 0 \\ 0 & \sqrt{1-\gamma}\end{pmatrix}, \quad K_1 = \begin{pmatrix}0 & \sqrt{\gamma} \\ 0 & 0\end{pmatrix}$$

In [ ]:
# Simulación de la relajación T1
T1 = 1.0
t_vals = np.linspace(0, 4*T1, 200)
rho_1 = np.array([[0,0],[0,1]], dtype=complex)  # estado |1⟩

pop_excited  = []  # ⟨1|ρ(t)|1⟩
pop_coherence = [] # |⟨0|ρ(t)|1⟩|, comenzando desde |+⟩

for t in t_vals:
    gamma = 1 - np.exp(-t/T1)
    rho_t_from1 = apply_kraus(rho_1, amplitude_damping_kraus(gamma))
    rho_t_from_plus = apply_kraus(rho_plus, amplitude_damping_kraus(gamma))
    pop_excited.append(rho_t_from1[1,1].real)
    pop_coherence.append(abs(rho_t_from_plus[0,1]))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(t_vals/T1, pop_excited, 'darkorange', lw=2, label='⟨1|ρ|1⟩')
axes[0].plot(t_vals/T1, 1 - np.array(pop_excited), 'steelblue', lw=2, label='⟨0|ρ|0⟩')
axes[0].set_xlabel('t / T₁'); axes[0].set_ylabel('Población')
axes[0].set_title('Relajación desde |1⟩'); axes[0].legend()

axes[1].plot(t_vals/T1, pop_coherence, 'mediumseagreen', lw=2, label='|ρ₀₁(t)|')
axes[1].plot(t_vals/T1, 0.5*np.exp(-t_vals/(2*T1)), 'k--', lw=1.5, label='0.5 exp(-t/2T₁)')
axes[1].set_xlabel('t / T₁'); axes[1].set_ylabel('Coherencia |ρ₀₁|')
axes[1].set_title('Decaimiento de coherencia (T₁)'); axes[1].legend()

plt.tight_layout(); plt.show()
print(f"Tiempo T₁: población excitada cae a 1/e ≈ {1/np.e:.3f}")

## 3. Canal de Decoherencia Pura (T₂)

El amortiguamiento de fase modela la **decoherencia pura** sin relajación energética ($T_2^*$). Las poblaciones diagonales de $\rho$ no cambian, pero las coherencias decaen:

$$\rho(t) = \begin{pmatrix}\rho_{00} & \rho_{01}e^{-t/T_2} \\ \rho_{10}e^{-t/T_2} & \rho_{11}\end{pmatrix}$$

En un qubit real: $1/T_2 = 1/(2T_1) + 1/T_\phi$ (donde $T_\phi$ es la decoherencia pura).

In [ ]:
# Comparación T1 vs T2* decay
T2 = 0.5  # T2 < 2*T1 (siempre)

coherences_T1 = []
coherences_T2 = []

for t in t_vals:
    gamma_T1 = 1 - np.exp(-t/T1)
    lam_T2   = 1 - np.exp(-t/T2)
    r_T1 = apply_kraus(rho_plus, amplitude_damping_kraus(gamma_T1))
    r_T2 = apply_kraus(rho_plus, phase_damping_kraus(lam_T2))
    coherences_T1.append(abs(r_T1[0,1]))
    coherences_T2.append(abs(r_T2[0,1]))

plt.figure(figsize=(7, 4))
plt.plot(t_vals/T1, coherences_T1, 'darkorange', lw=2, label=f'Amort. amplitud (T₁={T1})')
plt.plot(t_vals/T1, coherences_T2, 'steelblue',  lw=2, label=f'Amort. fase (T₂*={T2})')
plt.xlabel('t / T₁', fontsize=12)
plt.ylabel('|ρ₀₁(t)| (coherencia)', fontsize=12)
plt.title('Decaimiento de Coherencia: T₁ vs T₂*')
plt.legend(fontsize=10); plt.tight_layout(); plt.show()

print("Resumen canales cuánticos:")
for name, K, p_test in [('Depolarizante', depolarizing_kraus(0.2), 0.2),
                         ('Amp. damping',  amplitude_damping_kraus(0.2), 0.2),
                         ('Phase damping', phase_damping_kraus(0.2), 0.2)]:
    rho_out = apply_kraus(rho_plus, K)
    dm = DensityMatrix(rho_out)
    fid = float(state_fidelity(DensityMatrix(rho_plus), dm))
    print(f"  {name:20s}: pureza={float(dm.purity()):.4f}, fidelidad={fid:.4f}")